In [ ]:
#konfiguracija
import os, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

#kad rezultatai butu vienodi po kiekvieno paleidimo
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

#datos
TRAIN_START = pd.Timestamp("2020-04-01")
BLIND_START = pd.Timestamp("2026-01-01")
BLIND_END  = pd.Timestamp("2026-03-31")

#failai
CSV_PATH  = "duomenys.csv"
LOG_PATH  = "SARIMAX_Zurnalas.csv"
PARAMS_PATH = "SARIMAX_best_params.json"

#paskutine diena su faktais
_raw_csv = pd.read_csv(CSV_PATH)
_raw_csv.columns = ['Data'] + list(_raw_csv.columns[1:])
_raw_csv['Data'] = pd.to_datetime(_raw_csv['Data'], dayfirst=True)
_col_b = _raw_csv.iloc[:, 1]
_mask = _col_b.notna() & (_col_b != '') & (_col_b != 0)
TRAIN_END = pd.Timestamp(_raw_csv.loc[_mask, 'Data'].max())
del _raw_csv, _col_b, _mask
print(f'Mokymo periodas: {TRAIN_START.date()} -> {TRAIN_END.date()}')

#sarimax bendri parametrai
SEASONAL_PERIOD = 7     #savaitinis sezoniskumas
FOURIER_K    = 3     #fourier nariu metiniam ciklui
ALPHA_CI    = 0.10    #90% pasikliautini intervalai

#metrikos
def mae(y, yhat): return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))
def rmse(y, yhat): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat))**2)))
def mape(y, yhat):
  y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
  mask = y != 0
  return float(np.mean(np.abs((y[mask] - yhat[mask]) / y[mask])) * 100) if mask.any() else np.nan
def smape(y, yhat):
  y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
  denom = (np.abs(y) + np.abs(yhat)) / 2.0
  mask = denom != 0
  return float(np.mean(np.abs(y[mask] - yhat[mask]) / denom[mask]) * 100) if mask.any() else np.nan
def mase(y, yhat, y_hist, m=SEASONAL_PERIOD):
  y = np.asarray(y, dtype=float); yhat = np.asarray(yhat, dtype=float)
  y_hist = np.asarray(y_hist, dtype=float)
  denom = float(np.mean(np.abs(y_hist[m:] - y_hist[:-m])))
  return float(np.mean(np.abs(y - yhat)) / denom) if denom > 0 else np.nan

def savaites_nr_metuose(data: pd.Timestamp) -> int:
  jan1 = pd.Timestamp(year=data.year, month=1, day=1)
  pirmas_sekm = jan1 + pd.Timedelta(days=(6 - jan1.dayofweek) % 7)
  if data <= pirmas_sekm:
    return 1
  return 2 + (data - (pirmas_sekm + pd.Timedelta(days=1))).days // 7

print(f"Konfig: mokom {TRAIN_START.date()} -> {TRAIN_END.date()}, blind {BLIND_START.date()} -> {BLIND_END.date()}")
print(f"SARIMAX: s={SEASONAL_PERIOD}, Fourier K={FOURIER_K}, CI alpha={ALPHA_CI}")

#nutrinam senus parametrus kad modelis is naujo apsimokytu
if os.path.exists(PARAMS_PATH):
  os.remove(PARAMS_PATH)
  print(f"Istrintas: {PARAMS_PATH}")


In [ ]:
#duomenys ir egzogeniniai kintamieji
#sarimax sezoniskumui naudoja s=7, todel savaites dienos neitraukiamos kaip egz kintamieji
#tai daroma kad nesidubliuotu ta pati informaija

#nuskaitom csv ir uzpildom dienas
df_raw = pd.read_csv(CSV_PATH)
df_raw["Data"] = pd.to_datetime(df_raw["Data"], dayfirst=True, errors="coerce")
df_raw = df_raw.dropna(subset=["Data"]).sort_values("Data").set_index("Data")

paskutine_CSV = df_raw.index.max()
pilnas_indeksas = pd.date_range(TRAIN_START, paskutine_CSV, freq="D")
df = df_raw.reindex(pilnas_indeksas)
df.index.name = "Data"
print(f"Duomenys: {TRAIN_START.date()} -> {paskutine_CSV.date()} ({len(df)} d., truksta: {int(df['Skambuciai'].isna().sum())})")

#trukstamu reiksmiu uzpildymas
df["Skambuciai"] = df["Skambuciai"].interpolate(method="linear", limit=3, limit_area="inside")
for dt in df.index[df["Skambuciai"].isna()]:
  kand = []
  for y in range(TRAIN_START.year, dt.year):
    past_date = dt - pd.DateOffset(years=(dt.year - y))
    if past_date in df.index and not pd.isna(df.at[past_date, "Skambuciai"]):
      kand.append(float(df.at[past_date, "Skambuciai"]))
  if kand:
    df.at[dt, "Skambuciai"] = float(np.median(kand))
df["Skambuciai"] = df["Skambuciai"].fillna(df["Skambuciai"].shift(1).rolling(7, min_periods=1).mean())

#anomalijos tik mokymo lange
tm = df.loc[(df.index >= TRAIN_START) & (df.index <= TRAIN_END), "Skambuciai"]
sv = tm.shift(1).rolling(30, min_periods=7).mean()
ss = tm.shift(1).rolling(30, min_periods=7).std()
anomalijos = (((tm - sv).abs() > 3 * ss) & ss.notna()).sum()
print(f"Anomaliju kandidatai (>3 sigma): {int(anomalijos)}")

#LT sventes
LT_SVENTES = pd.to_datetime([
  "2020-01-01","2021-01-01","2022-01-01","2023-01-01","2024-01-01","2025-01-01","2026-01-01",
  "2020-02-16","2021-02-16","2022-02-16","2023-02-16","2024-02-16","2025-02-16","2026-02-16",
  "2020-03-11","2021-03-11","2022-03-11","2023-03-11","2024-03-11","2025-03-11","2026-03-11",
  "2020-04-12","2021-04-04","2022-04-17","2023-04-09","2024-03-31","2025-04-20","2026-04-05",
  "2020-04-13","2021-04-05","2022-04-18","2023-04-10","2024-04-01","2025-04-21","2026-04-06",
  "2020-05-01","2021-05-01","2022-05-01","2023-05-01","2024-05-01","2025-05-01","2026-05-01",
  "2020-06-24","2021-06-24","2022-06-24","2023-06-24","2024-06-24","2025-06-24","2026-06-24",
  "2020-07-06","2021-07-06","2022-07-06","2023-07-06","2024-07-06","2025-07-06","2026-07-06",
  "2020-08-15","2021-08-15","2022-08-15","2023-08-15","2024-08-15","2025-08-15","2026-08-15",
  "2020-11-01","2021-11-01","2022-11-01","2023-11-01","2024-11-01","2025-11-01","2026-11-01",
  "2020-11-02","2021-11-02","2022-11-02","2023-11-02","2024-11-02","2025-11-02","2026-11-02",
  "2020-12-24","2021-12-24","2022-12-24","2023-12-24","2024-12-24","2025-12-24","2026-12-24",
  "2020-12-25","2021-12-25","2022-12-25","2023-12-25","2024-12-25","2025-12-25","2026-12-25",
  "2020-12-26","2021-12-26","2022-12-26","2023-12-26","2024-12-26","2025-12-26","2026-12-26",
])
VELYKOS = pd.to_datetime(["2020-04-12","2021-04-04","2022-04-17","2023-04-09","2024-03-31","2025-04-20","2026-04-05"])
SEKMINES = pd.to_datetime(["2020-05-31","2021-05-23","2022-06-05","2023-05-28","2024-05-19","2025-06-08","2026-05-24"])
SVENT_SET = set(LT_SVENTES); VEL_SET = set(VELYKOS); SEK_SET = set(SEKMINES)

#egzogeniniai kintamieji - tik sventes ir fourier metiniam ciklui
def sukurti_exog(idx: pd.DatetimeIndex) -> pd.DataFrame:
  idx_s = pd.Series(idx, index=idx)
  exog = pd.DataFrame(index=idx)
  #sventiniai indikatoriai
  exog["is_holiday"]     = idx_s.isin(SVENT_SET).astype(int).values
  exog["day_before_holiday"] = (idx_s + pd.Timedelta(days=1)).isin(SVENT_SET).astype(int).values
  exog["day_after_holiday"] = (idx_s - pd.Timedelta(days=1)).isin(SVENT_SET).astype(int).values
  exog["is_easter"]     = idx_s.isin(VEL_SET).astype(int).values
  exog["is_pentecost"]    = idx_s.isin(SEK_SET).astype(int).values
  #fourier metiniam ciklui
  doy = idx.dayofyear.values.astype(float)
  period = 365.25
  for k in range(1, FOURIER_K + 1):
    exog[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * doy / period)
    exog[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * doy / period)
  return exog

#pilna egz matrica visam laikotarpiui
full_idx = pd.date_range(TRAIN_START, BLIND_END, freq="D")
df_exog_full = sukurti_exog(full_idx)
EXOG_COLS = list(df_exog_full.columns)
print(f"Egz kintamieji ({len(EXOG_COLS)}): {EXOG_COLS}")

#endogenine eilute
y_full = df["Skambuciai"].astype(float)
print(f"y: {y_full.index.min().date()} -> {y_full.index.max().date()} ({len(y_full.dropna())} d. su reiksmem)")


In [ ]:
#stacionarumas, transformacija, baziniai modeliai

from statsmodels.tsa.stattools import adfuller, kpss
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.filterwarnings('ignore', category=InterpolationWarning)

#mokymo eilute
y_train = y_full.loc[TRAIN_START:TRAIN_END].dropna()
print(f"Mokymo eilute: {y_train.index.min().date()} -> {y_train.index.max().date()} ({len(y_train)} d.)")

#log transformacija (skambuciai gali but 0)
y_train_log = np.log1p(y_train)
print(f"Log: min={y_train_log.min():.2f}, max={y_train_log.max():.2f}, std={y_train_log.std():.3f}")

#adf + kpss
def stac_testai(x: pd.Series, pav: str):
  adf_stat, adf_p, *_ = adfuller(x.dropna(), autolag="AIC")
  kpss_stat, kpss_p, *_ = kpss(x.dropna(), regression="c", nlags="auto")
  print(f" {pav:30s} ADF p={adf_p:.4f} KPSS p={kpss_p:.4f}")
  return adf_p, kpss_p

print("\n=== Stacionarumo testai ===")
print("ADF: nestacionari, KPSS: stacionari")
print("Stacionarumui reikia: ADF p<0.05 IR KPSS p>0.05")
stac_testai(y_train_log,           "log(1+y)")
stac_testai(y_train_log.diff(),        "log(1+y) diff(1)")
stac_testai(y_train_log.diff(SEASONAL_PERIOD),"log(1+y) diff(7)")
stac_testai(y_train_log.diff().diff(SEASONAL_PERIOD), "log(1+y) diff(1)+diff(7)")

#baziniai modeliai 2026 metams
y_blind = y_full.loc[BLIND_START:BLIND_END].dropna()
hist_tr = y_train.values.astype(float)

if len(y_blind):
  y_cat = y_full.dropna().copy()
  baziniai = {
    "Naive (t-1)":      y_cat.shift(1).loc[y_blind.index].values,
    "Seasonal naive (t-7)":  y_cat.shift(7).loc[y_blind.index].values,
    "7d slenk. vidurkis":   y_cat.shift(1).rolling(7).mean().loc[y_blind.index].values,
    "Same week last year":  y_cat.shift(364).loc[y_blind.index].values,
  }
  y_true = y_blind.values.astype(float)
  print(f"\n=== Baziniai modeliai 2026 m. ({len(y_blind)} d.) ===")
  print(f'{"Modelis":26s} {"MAE":>8s} {"RMSE":>8s} {"MAPE%":>8s} {"MASE":>8s}')
  for pav, yp in baziniai.items():
    yp_arr = np.asarray(yp, dtype=float)
    msk = ~np.isnan(yp_arr)
    if msk.any():
      print(f"{pav:26s} {mae(y_true[msk], yp_arr[msk]):8.1f} "
         f"{rmse(y_true[msk], yp_arr[msk]):8.1f} "
         f"{mape(y_true[msk], yp_arr[msk]):8.2f} "
         f"{mase(y_true[msk], yp_arr[msk], hist_tr):8.2f}")
else:
  print("\n2026 faktu dar nera.")

#acf/pacf grafikai
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig_acf, axes_acf = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_train.dropna(), lags=40, ax=axes_acf[0], title='ACF')
plot_pacf(y_train.dropna(), lags=40, ax=axes_acf[1], title='PACF')
plt.tight_layout()
plt.show()
print('ACF/PACF grafikai nubrezti.')

#box-cox
from scipy.stats import boxcox as sp_boxcox
try:
  _y_pos = y_train.dropna().values
  _y_pos = _y_pos[_y_pos > 0]
  _, bc_lambda = sp_boxcox(_y_pos)
  print(f'Box-Cox lambda: {bc_lambda:.4f}')
  if abs(bc_lambda) < 0.05:
    print(f' lambda ~ 0 -> log transformacija optimali.')
  elif abs(bc_lambda - 1.0) < 0.05:
    print(f' lambda ~ 1 -> transformacija nereikalinga.')
  else:
    print(f' lambda = {bc_lambda:.4f} -> tinka Box-Cox({bc_lambda:.2f}).')
except Exception as e:
  print(f' Box-Cox: {e}')


In [ ]:
#sarimax parametru paieska su auto_arima (vyksta vieną karta, paskui issaugoma atmintyje)

if os.path.exists(PARAMS_PATH):
  with open(PARAMS_PATH, "r", encoding="utf-8") as f:
    best_params = json.load(f)
  print(f"Parametrai pakrauti is {PARAMS_PATH}")
  print(f"  order={tuple(best_params['order'])}, seasonal_order={tuple(best_params['seasonal_order'])}")
else:
  print("Paleidziam auto_arima paieska...")
  try:
    import pmdarima as pm
  except ImportError as e:
    raise RuntimeError("Reikalinga pmdarima: pip install pmdarima") from e

  y_in = y_train_log.values
  X_in = df_exog_full.loc[y_train.index].values

  model_auto = pm.auto_arima(
    y_in, X=X_in,
    start_p=0, start_q=0, max_p=3, max_q=3,
    d=None, max_d=2,
    seasonal=True, m=SEASONAL_PERIOD,
    start_P=0, start_Q=0, max_P=2, max_Q=2,
    D=None, max_D=1,
    stepwise=True, suppress_warnings=True,
    error_action="ignore", information_criterion="aic",
    trace=False, random_state=SEED,
  )
  best_params = {
    "order":     list(model_auto.order),
    "seasonal_order": list(model_auto.seasonal_order),
    "aic":      float(model_auto.aic()),
    "bic":      float(model_auto.bic()),
  }
  with open(PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)
  print(f"Paieska baigta. order={tuple(best_params['order'])}, seasonal_order={tuple(best_params['seasonal_order'])}")
  print(f"  AIC={best_params['aic']:.1f} BIC={best_params['bic']:.1f}")

ORDER     = tuple(best_params["order"])
SEASONAL_ORDER = tuple(best_params["seasonal_order"])
print(f"\nSARIMAX: ARIMA{ORDER} x SARIMA{SEASONAL_ORDER[:-1]}[{SEASONAL_ORDER[-1]}] + {len(EXOG_COLS)} egz kint")


In [ ]:
#mokymas, savaitine prognoze, zurnalas

from statsmodels.tsa.statespace.sarimax import SARIMAX

#mokymo aibe iki paskutines turimos dienos
y_train = y_full.loc[TRAIN_START:].dropna()
paskutine_turima = y_train.index.max()
X_moky = df_exog_full.loc[y_train.index].values
print(f"Mokymo rinkinys: {y_train.index.min().date()} -> {paskutine_turima.date()} ({len(y_train)} d.)")

#log transformacija
y_train_t = np.log1p(y_train.values)

#mokymas
modelis = SARIMAX(
  endog=y_train_t,
  exog=X_moky,
  order=ORDER,
  seasonal_order=SEASONAL_ORDER,
  enforce_stationarity=False,
  enforce_invertibility=False,
)
rez = modelis.fit(disp=False, maxiter=500)
print(f"SARIMAX apmokytas: LogLik={rez.llf:.1f} AIC={rez.aic:.1f} BIC={rez.bic:.1f}")

def kita_savaite(paskutine: pd.Timestamp):
  #grazina pir-sek savaite (arba galo gabalą)
  start = paskutine + pd.Timedelta(days=1)
  if start > BLIND_END:
    return None
  days_until_sunday = 6 - start.dayofweek
  week_end = start + pd.Timedelta(days=days_until_sunday)
  week_end = min(week_end, BLIND_END)
  return pd.date_range(start, week_end, freq="D")

def prognozuoti_savaite(fit_rez, savaite: pd.DatetimeIndex):
  #n-zingsniu i prieki + 90% CI
  X_ateitis = df_exog_full.loc[savaite].values
  fc = fit_rez.get_forecast(steps=len(savaite), exog=X_ateitis)
  mean_t = fc.predicted_mean
  ci_t  = fc.conf_int(alpha=ALPHA_CI)
  #atvirkstine log1p
  mean_y = np.expm1(mean_t)
  ci_low = np.maximum(0.0, np.expm1(ci_t[:, 0]))
  ci_high = np.maximum(0.0, np.expm1(ci_t[:, 1]))
  return pd.DataFrame({
    "Prognoze": mean_y,
    "CI_Lower": ci_low,
    "CI_Upper": ci_high,
  }, index=savaite)

#zurnalas
STULP = ["Savaites_Nr", "Prognoze", "CI_Lower", "CI_Upper", "Faktas", "Abs_Klaida", "Proc_Klaida"]
if os.path.exists(LOG_PATH):
  zurnalas = pd.read_csv(LOG_PATH, index_col="Data", parse_dates=True)
  for c in STULP:
    if c not in zurnalas.columns:
      zurnalas[c] = np.nan
  zurnalas = zurnalas[STULP]
else:
  zurnalas = pd.DataFrame(columns=STULP)
  zurnalas.index = pd.DatetimeIndex([], name="Data")
print(f"Zurnale: {len(zurnalas)} irasu")

#uzpildom faktais
uzpildyta = 0
for dt in zurnalas.index:
  if pd.isna(zurnalas.at[dt, "Faktas"]) and dt in y_full.index \
      and not pd.isna(y_full.at[dt]):
    f = float(y_full.at[dt])
    p = float(zurnalas.at[dt, "Prognoze"])
    zurnalas.at[dt, "Faktas"]   = f
    zurnalas.at[dt, "Abs_Klaida"] = abs(f - p)
    zurnalas.at[dt, "Proc_Klaida"] = abs(f - p) / f * 100 if f != 0 else np.nan
    uzpildyta += 1
if uzpildyta:
  print(f"Uzpildyta {uzpildyta} faktu")

#paskutines savaites paklaida
uzbaigta = zurnalas.dropna(subset=["Faktas"])
if len(uzbaigta):
  pask_nr = int(uzbaigta["Savaites_Nr"].max())
  grp = uzbaigta[uzbaigta["Savaites_Nr"] == pask_nr]
  if len(grp) >= 1:
    y_f = grp["Faktas"].values.astype(float)
    y_p = grp["Prognoze"].values.astype(float)
    print(f"\n--- Praejusi savaite {pask_nr}: {grp.index.min().date()} -> {grp.index.max().date()} ({len(grp)} d.) ---")
    print(f" MAE : {mae(y_f, y_p):8.1f}")
    print(f" RMSE : {rmse(y_f, y_p):8.1f}")
    print(f" MAPE : {mape(y_f, y_p):8.2f}%")
    print(f" MASE : {mase(y_f, y_p, y_train.values):8.2f}")

#nauja prognoze
savaite = kita_savaite(paskutine_turima)
if savaite is None:
  print("\n2026 prognozavimas baigtas (pasieke 2026-03-31).")
else:
  pirm, sekm = savaite[0], savaite[-1]
  sav_nr = savaites_nr_metuose(pirm)
  print(f"\nProgostozuojam savaite {sav_nr} ({pirm.year}): {pirm.date()} -> {sekm.date()} ({len(savaite)} d.)")
  prog_df = prognozuoti_savaite(rez, savaite)
  for dt in savaite:
    zurnalas.loc[dt] = {
      "Savaites_Nr": sav_nr,
      "Prognoze":  int(round(prog_df.at[dt, "Prognoze"])),
      "CI_Lower":  int(round(prog_df.at[dt, "CI_Lower"])),
      "CI_Upper":  int(round(prog_df.at[dt, "CI_Upper"])),
      "Faktas":   np.nan,
      "Abs_Klaida": np.nan,
      "Proc_Klaida": np.nan,
    }
  print("\nDienos prognozes (su 90% CI):")
  for dt in savaite:
    diena = ["Pir","Ant","Tre","Ket","Pen","Ses","Sek"][dt.dayofweek]
    p = int(round(prog_df.at[dt, "Prognoze"]))
    lo = int(round(prog_df.at[dt, "CI_Lower"]))
    hi = int(round(prog_df.at[dt, "CI_Upper"]))
    print(f" {dt.date()} ({diena}): {p:5d}  [90% CI: {lo:5d} - {hi:5d}]")

zurnalas.index.name = "Data"
zurnalas.sort_index(inplace=True)
zurnalas.to_csv(LOG_PATH)
print(f"\nZurnalas issaugotas: {LOG_PATH}")


In [ ]:
#grafikai, diagnostika, metrikos, adaptacija

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import scipy.stats as sp_stats

#sarimax vs baseline vs faktas (2026 q1)
dates_2026 = zurnalas[zurnalas.index >= "2026-01-01"].index

if len(dates_2026) > 0:
  #istorija + prognozes (kad baseline lag veiktu)
  y_tmp = y_full.copy()
  for dt in dates_2026:
    if dt not in y_tmp.index or pd.isna(y_tmp.get(dt, np.nan)):
      y_tmp.loc[dt] = zurnalas.at[dt, "Prognoze"]
  y_tmp = y_tmp.sort_index()

  atsarginis_vidurkis = y_full.mean()
  baseline_vals = {
    "Naive (t-1)":   y_tmp.shift(1).loc[dates_2026].fillna(atsarginis_vidurkis).values,
    "Seasonal (t-7)":  y_tmp.shift(7).loc[dates_2026].fillna(atsarginis_vidurkis).values,
    "SameYear (t-364)": y_tmp.shift(364).loc[dates_2026].fillna(atsarginis_vidurkis).values,
  }

  sarimax_pred = zurnalas.loc[dates_2026, "Prognoze"].values.astype(float)
  ci_lo    = zurnalas.loc[dates_2026, "CI_Lower"].values.astype(float)
  ci_hi    = zurnalas.loc[dates_2026, "CI_Upper"].values.astype(float)
  faktas_vals = zurnalas.loc[dates_2026, "Faktas"].values

  fig, axes = plt.subplots(2, 1, figsize=(15, 10))

  #grafikas 1
  ax = axes[0]
  colors = ["orange", "green", "red"]
  for (name, vals), color in zip(baseline_vals.items(), colors):
    ax.plot(dates_2026, vals, label=name, linewidth=1.5, alpha=0.6, color=color, linestyle="--")

  ax.fill_between(dates_2026, ci_lo, ci_hi, color="royalblue", alpha=0.18, label="SARIMAX 90% CI")
  ax.plot(dates_2026, sarimax_pred, label="SARIMAX", linewidth=2.5, color="royalblue", marker="o", markersize=4, zorder=3)

  if not pd.isna(faktas_vals).all():
    ax.plot(dates_2026, faktas_vals, label="Faktas", linewidth=3, color="black", marker="s", markersize=5, zorder=5)

  ax.set_xlabel("Data (2026)", fontsize=11)
  ax.set_ylabel("Skambuciu kiekis", fontsize=11)
  ax.set_title(f"SARIMAX vs baseline vs faktas (iki {dates_2026.max().date()})", fontsize=13, fontweight="bold")
  ax.legend(loc="best", fontsize=10)
  ax.grid(True, alpha=0.3)
  ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
  ax.xaxis.set_minor_locator(mdates.DayLocator())
  ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
  plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

  #grafikas 2 - kaupiama suma
  ax = axes[1]
  for (name, vals), color in zip(baseline_vals.items(), colors):
    ax.plot(dates_2026, np.cumsum(vals), label=f"{name} (kumul.)", linewidth=1.5, alpha=0.6, color=color, linestyle="--")

  ax.plot(dates_2026, np.cumsum(sarimax_pred), label="SARIMAX (kumul.)", linewidth=2.5, color="royalblue", marker="o", markersize=4, zorder=3)

  if not pd.isna(faktas_vals).all():
    faktas_cs = pd.Series(faktas_vals).cumsum()
    ax.plot(dates_2026, faktas_cs, label="Faktas (kumul.)", linewidth=3, color="black", marker="s", markersize=5, zorder=5)

  ax.set_xlabel("Data (2026)", fontsize=11)
  ax.set_ylabel("Is viso skambuciu", fontsize=11)
  ax.set_title(f"Srauto augimas (iki {dates_2026.max().date()})", fontsize=13, fontweight="bold")
  ax.legend(loc="best", fontsize=10)
  ax.grid(True, alpha=0.3)
  ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
  ax.xaxis.set_minor_locator(mdates.DayLocator())
  ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
  plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")

  plt.tight_layout()
  plt.show()

  #suvestine
  print("\n" + "="*70)
  print(f"SUVESTINE (iki {dates_2026.max().date()})")
  print("="*70)
  print(f"\n{'Modelis':<25} {'Vidut./d':>12} {'Viso':>12} {'Min':>8} {'Max':>8}")
  print("-" * 70)

  if not pd.isna(faktas_vals).all():
    vf = faktas_vals[~pd.isna(faktas_vals)]
    print(f"{'FAKTAS':<25} {vf.mean():>12.0f} {vf.sum():>12.0f} {vf.min():>8.0f} {vf.max():>8.0f}")
    print("-" * 70)

  print(f"{'SARIMAX':<25} {sarimax_pred.mean():>12.0f} {sarimax_pred.sum():>12.0f} {sarimax_pred.min():>8.0f} {sarimax_pred.max():>8.0f}")

  for name, vals in baseline_vals.items():
    print(f"{name:<25} {vals.mean():>12.0f} {vals.sum():>12.0f} {vals.min():>8.0f} {vals.max():>8.0f}")
  print("="*70)

else:
  print("Zurnale dar nera 2026 prognoziu.")

#winkler funkcija
def winkler_score(y_true, ci_lower, ci_upper, alpha=0.10):
  n = len(y_true)
  total = 0.0
  for i in range(n):
    width = ci_upper[i] - ci_lower[i]
    if y_true[i] < ci_lower[i]:
      total += width + (2.0 / alpha) * (ci_lower[i] - y_true[i])
    elif y_true[i] > ci_upper[i]:
      total += width + (2.0 / alpha) * (y_true[i] - ci_upper[i])
    else:
      total += width
  return total / n

#bendros 2026 metrikos
uzbaigta = zurnalas.dropna(subset=["Faktas"]).copy()
if len(uzbaigta) == 0:
  print("\n2026 faktu dar nera.")
else:
  y_f = uzbaigta["Faktas"].values.astype(float)
  y_p = uzbaigta["Prognoze"].values.astype(float)
  hist = y_train.values.astype(float)
  print(f"\n=== 2026 m bendros metrikos ({len(uzbaigta)} d.) ===")
  print(f" MAE  : {mae(y_f, y_p):8.2f}")
  print(f" RMSE : {rmse(y_f, y_p):8.2f}")
  print(f" MAPE : {mape(y_f, y_p):8.2f}%")
  print(f" sMAPE : {smape(y_f, y_p):8.2f}%")
  print(f" MASE : {mase(y_f, y_p, hist):8.2f}")

  #picp
  lo = uzbaigta["CI_Lower"].values.astype(float)
  hi = uzbaigta["CI_Upper"].values.astype(float)
  picp = float(np.mean((y_f >= lo) & (y_f <= hi)))
  pinaw = float(np.mean(hi - lo) / (hist.max() - hist.min()))
  print(f" PICP : {picp*100:6.1f}% (tikslas ~90%)")
  print(f" PINAW : {pinaw:8.3f}")
  ws = winkler_score(y_f, lo, hi, alpha=ALPHA_CI)
  print(f" Winkler : {ws:8.2f}")

#likuciu diagnostika
try:
  from statsmodels.stats.diagnostic import acorr_ljungbox
  resid = rez.resid
  fig, axes = plt.subplots(1, 2, figsize=(14, 4))
  resid_plot = resid[SEASONAL_PERIOD:]
  plot_index = y_train.index[SEASONAL_PERIOD:]

  axes[0].plot(plot_index, resid_plot, color="steelblue", linewidth=0.8)
  axes[0].axhline(0, color="black", linewidth=0.5)
  axes[0].set_title("SARIMAX likuciai (log skale)")
  axes[0].grid(True, alpha=0.3)

  axes[1].hist(resid_plot, bins=50, color="steelblue", edgecolor="white")
  axes[1].set_title("Likuciu histograma")
  axes[1].grid(True, alpha=0.3)

  plt.tight_layout()
  plt.show()

  lb = acorr_ljungbox(resid, lags=[14], return_df=True)
  print(f"\nLjung-Box (lag=14): Q={lb['lb_stat'].iloc[0]:.2f}, p={lb['lb_pvalue'].iloc[0]:.4f} "
     f"-> {'baltas triuksmas' if lb['lb_pvalue'].iloc[0] > 0.05 else 'yra autokoreliaciju'}")
except Exception as e:
  print(f"Diagnostikos nepavyko: {e}")

# adaptacijos kreive ir testai
if len(uzbaigta) >= 7:
  uzbaigta["Proc_Klaida"] = np.abs(uzbaigta["Faktas"] - uzbaigta["Prognoze"]) / uzbaigta["Faktas"] * 100
  sav_mape = uzbaigta.groupby("Savaites_Nr")["Proc_Klaida"].mean()

  fig, ax = plt.subplots(figsize=(10, 5))
  ax.plot(sav_mape.index.astype(int), sav_mape.values, "o-", color="crimson", linewidth=2, label="Savaites MAPE")
  ax.axhline(10, color="green", linestyle="--", label="10% riba")
  ax.set_xlabel("2026 m savaites nr.")
  ax.set_ylabel("MAPE (%)")
  ax.set_title("SARIMAX adaptacija po reformos")
  ax.grid(True, alpha=0.3)
  ax.legend()

  plt.tight_layout()
  plt.show()

  # testai
  print("\n=== Adaptacija po reformos ===")
  savaites = sav_mape.index.values
  mape_reiksmes = sav_mape.values

  if len(sav_mape) >= 2:
    pirma_sav = uzbaigta[uzbaigta["Savaites_Nr"] == savaites[0]]["Proc_Klaida"].values
    pask_sav = uzbaigta[uzbaigta["Savaites_Nr"] == savaites[-1]]["Proc_Klaida"].values

    min_len = min(len(pirma_sav), len(pask_sav))
    if min_len > 0:
      stat_w, p_w = sp_stats.wilcoxon(pirma_sav[:min_len], pask_sav[:min_len], alternative='greater')
      print(f"Wilcoxon (pirma > paskutine): p={p_w:.4f}")
      if p_w < 0.05:
        print(" -> Atmesta: pradine paklaida didesne nei paskutine (yra adaptacija)")
      else:
        print(" -> Neatmesta: skirtumas nereiksmingas")

  if len(sav_mape) >= 3:
    stat_k, p_k = sp_stats.kendalltau(savaites, mape_reiksmes)
    print(f"Mann-Kendall trendo testas: p={p_k:.4f}, Tau={stat_k:.3f}")
    if p_k < 0.05 and stat_k < 0:
      print(" -> Atmesta: yra reiksmingas mape mazejimo trendas")
    else:
      print(" -> Neatmesta: nera mazejimo trendo")
else:
  print("\nReikia bent 1 pilnos savaites adaptacijos analizei.")


In [ ]:
#sarima vs sarimax palyginimas
#tikrinam ar egzogeniniai kintamieji svarbus

import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX as SM_SARIMAX

print('=' * 70)
print('SARIMA vs SARIMAX ')
print('=' * 70)

print('\nMokom SARIMA modeli (be egz kintamuju)...')
try:
  sarima_order = rez.specification['order']
  sarima_seasonal = rez.specification['seasonal_order']

  sarima_model = SM_SARIMAX(
    y_train,
    order=sarima_order,
    seasonal_order=sarima_seasonal,
    enforce_stationarity=False,
    enforce_invertibility=False
  )
  sarima_rez = sarima_model.fit(disp=False, maxiter=500)

  #aic/bic
  print('\n--- AIC / BIC ---')
  print(f" {'Modelis':<15} {'AIC':>12} {'BIC':>12}")
  print(f" {'-'*39}")
  print(f" {'SARIMAX':<15} {rez.aic:>12.2f} {rez.bic:>12.2f}")
  print(f" {'SARIMA':<15} {sarima_rez.aic:>12.2f} {sarima_rez.bic:>12.2f}")

  aic_diff = sarima_rez.aic - rez.aic
  bic_diff = sarima_rez.bic - rez.bic
  print(f'\n dAIC (SARIMA - SARIMAX) = {aic_diff:+.2f}')
  print(f' dBIC (SARIMA - SARIMAX) = {bic_diff:+.2f}')

  if aic_diff > 0 and bic_diff > 0:
    print(' -> SARIMAX geresnis pagal abu kriterijus')
    print(' -> Patvirtinta: egz kintamieji pagerina modeli.')
  elif aic_diff > 0:
    print(' -> SARIMAX geresnis pagal AIC, ne pagal BIC')
    print(' -> Dalinai patvirtinta.')
  elif bic_diff > 0:
    print(' -> SARIMAX geresnis pagal BIC, ne pagal AIC')
    print(' -> Dalinai patvirtinta.')
  else:
    print(' -> SARIMA geresnis')
    print(' -> Nepatvirtinta.')

  #likelihood ratio testas
  from scipy import stats as sp_stats
  lr_stat = 2 * (rez.llf - sarima_rez.llf)
  df_diff = rez.df_model - sarima_rez.df_model
  if df_diff > 0 and lr_stat > 0:
    lr_pval = 1 - sp_stats.chi2.cdf(lr_stat, df_diff)
    print(f'\n--- Likelihood-Ratio testas ---')
    print(f' LR statistika: {lr_stat:.4f}')
    print(f' Laisves laipsniai: {df_diff}')
    print(f' p-reiksme: {lr_pval:.6f}')
    if lr_pval < 0.05:
      print(' -> Egz kintamieji reiksmingai pagerina (p < 0.05)')
    else:
      print(' -> Egz kintamieji nereiksmingai pagerina (p >= 0.05)')

except Exception as e:
  print(f'Palyginimas nepavyko: {e}')


In [ ]:
#dm-hln testas: sarimax vs baziniai modeliai

import numpy as np
import pandas as pd
from scipy import stats as sp_stats

print('=' * 70)
print('DM-HLN TESTAS')
print('=' * 70)

def dm_hln_test(e1, e2, h=1, power=2):
  e1, e2 = np.asarray(e1, dtype=float), np.asarray(e2, dtype=float)
  T = len(e1)

  d = np.abs(e1)**power - np.abs(e2)**power
  d_bar = d.mean()

  gamma_0 = np.var(d, ddof=0)
  gamma_sum = 0.0
  for k in range(1, h):
    gamma_k = np.cov(d[k:], d[:-k], ddof=0)[0, 1]
    gamma_sum += gamma_k
  var_d = (gamma_0 + 2 * gamma_sum) / T

  if var_d <= 0:
    return np.nan, np.nan, np.nan, np.nan

  dm_stat = d_bar / np.sqrt(var_d)
  hln_factor = np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
  hln_stat = dm_stat * hln_factor

  p_dm = 2 * sp_stats.t.sf(np.abs(dm_stat), df=T-1)
  p_hln = 2 * sp_stats.t.sf(np.abs(hln_stat), df=T-1)

  return dm_stat, hln_stat, p_dm, p_hln

try:
  uzb = zurnalas.dropna(subset=['Faktas'])
  if len(uzb) >= 7:
    y_fact_idx = uzb.index
    y_fact = uzb['Faktas'].values.astype(float)
    y_pred = uzb['Prognoze'].values.astype(float)

    errors_sarimax = y_fact - y_pred

    y_cat = y_full.loc[:y_fact_idx.max()].copy()

    y_naive = y_cat.shift(1).loc[y_fact_idx].values
    y_snaive = y_cat.shift(7).loc[y_fact_idx].values
    y_ma7  = y_cat.shift(1).rolling(7).mean().loc[y_fact_idx].values
    y_sml  = y_cat.shift(364).loc[y_fact_idx].values

    benchmarks = {
      'Naive (t-1)':     y_fact - y_naive,
      'Seasonal Naive':   y_fact - y_snaive,
      '7-d slenk vidurkis': y_fact - y_ma7,
      'Same week last yr':  y_fact - y_sml,
    }

    print(f'\nDuomenu tasku: {len(y_fact)}')
    print(f'\n{"Benchmark":<20} {"DM-stat":>10} {"HLN-stat":>10} {"p(DM)":>10} {"p(HLN)":>10} {"Isvada":>20}')
    print(f'{"="*80}')

    results = []
    for name, e_bench in benchmarks.items():
      mask = ~np.isnan(e_bench)
      if mask.sum() < 7:
        continue

      dm_s, hln_s, p_dm, p_hln = dm_hln_test(
        e_bench[mask], errors_sarimax[mask], h=1, power=2
      )
      results.append((name, dm_s, hln_s, p_dm, p_hln))

      if np.isnan(p_hln):
        verdict = 'N/A'
      elif p_hln < 0.05 and dm_s > 0:
        verdict = 'SARIMAX geresnis'
      elif p_hln < 0.05 and dm_s < 0:
        verdict = 'Benchmark geresnis'
      else:
        verdict = 'Nesiskiria'

      print(f'{name:<20} {dm_s:>10.4f} {hln_s:>10.4f} {p_dm:>10.6f} {p_hln:>10.6f} {verdict:>20}')

    #holm-bonferroni
    print(f'\n--- Holm-Bonferroni korekcija ---')
    p_values = [(r[0], r[4]) for r in results if not np.isnan(r[4])]
    p_values.sort(key=lambda x: x[1])
    m = len(p_values)

    for rank, (name, p) in enumerate(p_values, 1):
      alpha_adj = 0.05 / (m - rank + 1)
      significant = 'Taip' if p < alpha_adj else 'Ne'
      print(f' {rank}. {name:<20} p={p:.6f} alpha_adj={alpha_adj:.6f} Reiksminga: {significant}')

  else:
    print('Per mazai uzbaigtu prognoziu (reikia >= 7).')

except Exception as e:
  print(f'DM-HLN testas nepavyko: {e}')
  import traceback; traceback.print_exc()


In [ ]:
#papildoma likuciu diagnostika - q-q, jarque-bera, arch lm

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

resid_full = np.asarray(rez.resid)
#praleidziam pradines dienas (filtro inicializacijos triuksmas)
resid_p6 = resid_full[SEASONAL_PERIOD * 4:]
print(f"Likuciu kiekis: {len(resid_p6)} d. (praleidus pirmas {SEASONAL_PERIOD * 4})")
print(f" vidurkis={resid_p6.mean():.4f} std={resid_p6.std():.4f}")
print(f" min={resid_p6.min():.3f} max={resid_p6.max():.3f}")

#q-q grafikas
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
stats.probplot(resid_p6, dist="norm", plot=axes[0])
axes[0].set_title("SARIMAX likuciu Q-Q")
axes[0].grid(True, alpha=0.3)

axes[1].hist(resid_p6, bins=60, density=True, color="steelblue", edgecolor="white", alpha=0.8, label="Likuciai")
xs = np.linspace(resid_p6.min(), resid_p6.max(), 200)
axes[1].plot(xs, stats.norm.pdf(xs, resid_p6.mean(), resid_p6.std()),
       "r-", linewidth=2, label=f"N({resid_p6.mean():.2f}, {resid_p6.std():.2f}^2)")
axes[1].set_title("Likuciu histograma + normalusis tankis")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

#jarque-bera
jb_stat, jb_p = stats.jarque_bera(resid_p6)
skew = float(stats.skew(resid_p6))
kurt = float(stats.kurtosis(resid_p6, fisher=True))
print(f"\nJarque-Bera: JB={jb_stat:.2f} p={jb_p:.4f}")
print(f" skew : {skew:7.3f}")
print(f" kurt : {kurt:7.3f}")
if jb_p > 0.05:
  print(" -> Neatmesta: likuciai normalus")
else:
  print(" -> Atmesta: likuciai NEIRA normalus")

#arch lm (heteroskedastiskumas)
try:
  from statsmodels.stats.diagnostic import het_arch
  arch_stat, arch_p, f_stat, f_p = het_arch(resid_p6, nlags=12)
  print(f"\nARCH LM (lag=12): LM={arch_stat:.2f} p={arch_p:.4f}")
  if arch_p > 0.05:
    print(" -> Neatmesta: nera arch efekto")
  else:
    print(" -> Atmesta: yra heteroskedastiskumas - svarstyti GARCH(1,1)")
except Exception as e:
  print(f"ARCH testas nepavyko: {e}")

#standartizuoti likuciai
resid_std = resid_p6 / resid_p6.std()
outliers = int(np.sum(np.abs(resid_std) > 3))
print(f"\nIsskirtys |z|>3: {outliers} d. ({100*outliers/len(resid_std):.1f}%)")


In [ ]:
#priedas a: papildomas vertinimas su atsitiktiniu padalijimu

#paimti po keliasdesimt dienu is kiekvienu metu kaip testus
#sarimax yra autoregresinis, todel klasinis random shuffle netinka
#cia naudojam in-sample 1-zingsnio prognozes atsitiktinai parinktose dienose

import numpy as np
import pandas as pd

DAYS_PER_YEAR = 60
RANDOM_STATE = SEED

#mokymo metu dienos
mok_idx = y_train.index
years = sorted(set(mok_idx.year))
print(f"Metai: {years}")

rng = np.random.default_rng(RANDOM_STATE)
test_dates_all = []
for yr in years:
  yr_dates = mok_idx[mok_idx.year == yr]
  if yr == years[0]:
    yr_dates = yr_dates[14:] #praleidziam pirmas dienas
  n = min(DAYS_PER_YEAR, len(yr_dates))
  chosen = rng.choice(yr_dates.values, size=n, replace=False)
  test_dates_all.extend(sorted(chosen))
test_dates = pd.DatetimeIndex(sorted(test_dates_all))
print(f"Testo dienu: {len(test_dates)}")

#in-sample prognozes
pred = rez.get_prediction(start=0, end=len(y_train)-1)
mean_t = np.asarray(pred.predicted_mean)
mean_y = np.expm1(mean_t)

pred_series = pd.Series(mean_y, index=y_train.index)
act_series = y_train

test_dates = test_dates.intersection(y_train.index)

y_f = act_series.loc[test_dates].values.astype(float)
y_p = pred_series.loc[test_dates].values.astype(float)

maeA = float(np.mean(np.abs(y_f - y_p)))
rmseA = float(np.sqrt(np.mean((y_f - y_p) ** 2)))
y_nz = y_f != 0
mapeA = float(np.mean(np.abs((y_f[y_nz] - y_p[y_nz]) / y_f[y_nz])) * 100) if y_nz.any() else np.nan
denom = (np.abs(y_f) + np.abs(y_p)) / 2.0
mk  = denom != 0
smapeA = float(np.mean(np.abs(y_f[mk] - y_p[mk]) / denom[mk]) * 100) if mk.any() else np.nan

print("\n=== Priedas A - rezultatai ===")
print(f" Testo d.: {len(test_dates)} (po ~{DAYS_PER_YEAR} is metu)")
print(f" MAE  : {maeA:8.2f}")
print(f" RMSE : {rmseA:8.2f}")
print(f" MAPE : {mapeA:8.2f}%")
print(f" sMAPE : {smapeA:8.2f}%")

#metrikos pagal metus
print("\nPagal metus:")
print(f" {'Metai':<6} {'n':>5} {'MAE':>8} {'MAPE%':>8}")
for yr in years:
  mask = test_dates.year == yr
  if mask.sum() == 0: continue
  yfy = y_f[mask]; ypy = y_p[mask]
  ma = float(np.mean(np.abs(yfy - ypy)))
  nz = yfy != 0
  mpe = float(np.mean(np.abs((yfy[nz] - ypy[nz]) / yfy[nz])) * 100) if nz.any() else np.nan
  print(f" {yr:<6} {int(mask.sum()):>5} {ma:>8.1f} {mpe:>8.2f}")

print("\nPastaba: in-sample metrikos optimistiniai - pagrindinis vertinimas lieka 2026 q1.")

#grafikai
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_f, y_p, alpha=0.4, s=15, color="steelblue", edgecolors="none")
mn, mx = min(y_f.min(), y_p.min()), max(y_f.max(), y_p.max())
ax.plot([mn, mx], [mn, mx], "k--", linewidth=1, label="Ideali (y=x)")
ax.set_xlabel("Faktas"); ax.set_ylabel("Prognoze")
ax.set_title(f"SARIMAX priedas A: faktas vs prognoze (n={len(y_f)})")
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
sort_order = np.argsort(test_dates)
td_sorted = test_dates[sort_order]
yf_sorted = y_f[sort_order]
yp_sorted = y_p[sort_order]
step = max(1, len(td_sorted) // 80)
ax.plot(range(len(yf_sorted[::step])), yf_sorted[::step], "k-", linewidth=1.5, label="Faktas", alpha=0.8)
ax.plot(range(len(yp_sorted[::step])), yp_sorted[::step], "-", color="steelblue", linewidth=1.5, label="SARIMAX", alpha=0.7)
ax.fill_between(range(len(yf_sorted[::step])), yf_sorted[::step], yp_sorted[::step], alpha=0.15, color="steelblue")
ax.set_xlabel("Testo dienos (chronologine)"); ax.set_ylabel("Skambuciai")
ax.set_title("SARIMAX priedas A: chronologine juosta")
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

#likuciu diagnostika
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

resid_appendix = y_f - y_p

n_test = min(len(resid_appendix), 5000)
sw_stat, sw_p = sp_stats.shapiro(resid_appendix[:n_test])
print(f'\n--- Shapiro-Wilk normalumo testas ---')
print(f' W={sw_stat:.4f}, p={sw_p:.6f}')
if sw_p > 0.05:
  print(' -> likuciai normalus')
else:
  print(' -> likuciai nukrypsta nuo normalumo (iprasta)')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(resid_appendix, bins=40, density=True, color='steelblue', edgecolor='white', alpha=0.8, label='Likuciai')
xs = np.linspace(resid_appendix.min(), resid_appendix.max(), 200)
axes[0].plot(xs, sp_stats.norm.pdf(xs, resid_appendix.mean(), resid_appendix.std()),
       'r-', linewidth=2, label=f'N({resid_appendix.mean():.1f}, {resid_appendix.std():.1f})')
axes[0].set_title('Priedas A: likuciu histograma')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

sp_stats.probplot(resid_appendix, dist='norm', plot=axes[1])
axes[1].set_title('Priedas A: Q-Q')
axes[1].grid(True, alpha=0.3)

axes[2].plot(range(len(resid_appendix)), resid_appendix, '-', color='steelblue', linewidth=0.8, alpha=0.7)
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[2].axhline(y=resid_appendix.mean() + 2*resid_appendix.std(), color='orange', linestyle=':', linewidth=1, label='2 sigma')
axes[2].axhline(y=resid_appendix.mean() - 2*resid_appendix.std(), color='orange', linestyle=':', linewidth=1)
axes[2].set_title('Priedas A: likuciai laike')
axes[2].set_xlabel('Stebejimo nr.')
axes[2].set_ylabel('Likana')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
